# Chapter 06 제출 답안. 데이터를 보며 질문을 만드는 EDA

## 0. 제출 정보

- 이름: 이성민
- GitHub ID: minilee0307-dot
- 작성·실행일: 2026-10-08
- 최종 Notebook URL(게시·제출 대상): https://github.com/minilee0307-dot/llm-data-analysis-study/blob/main/chapter06/chapter06.ipynb
- 실습 상태: **COMPLETE** — 전체 코드 셀 실행 및 결과 저장. GitHub 표시와 eTL 접수는 별도 확인 항목이다.
- 전체 제출 상태: **PARTIAL** — 실습은 완료했고 온라인 게시·접수 확인은 남아 있다.
- GitHub 게시·표시 상태: **확인 대기**
- eTL 제출 상태: **접수 확인 대기** — 상세 페이지에서 제출 시작 상태를 확인했다.

### 핵심 답변

**2025-07-02~2026-08-28(양 끝 포함)의 유효한 completed 주문 상세 금액은 134,523,000원이다.** 상세 421행, 주문 175건을 분석했다. 카테고리·월·도시·고객·결제수단·월×카테고리 합계가 모두 이 금액과 같았다. 금액 1위 카테고리는 **패션 32,940,000원(24.49%)**이다.

본 과제는 Chapter 05에서 보존한 **수업용 합성 데이터의 clean CSV 4개**로 실행했다. 고객수·상품수는 clean 전체를, 모든 구매금액은 `order_status == "completed"`이면서 주문일이 유효하고 2026-10-08 이하인 공통 범위를 썼다. 날짜 결측과 미래 후보의 제외 영향을 아래 모집단 표에 기록했다. 주문금액은 **정제 후 남은 유효 상세의 합계**다. 일부 상세가 격리된 주문이 분석에 18건 남아 있으므로 원래 주문의 전체 청구금액과 다를 수 있다. 금액은 `quantity × unit_price`이며 `products.price`로 대체하지 않았다.

공식 [Notebook](https://github.com/GilbertMoon/llm-data-analysis-course/blob/b1597e10c87f5c461bdb7e3f8f9b8454ba8cbedc/notebooks/ch06_eda_questions.ipynb), [실습 가이드](https://github.com/GilbertMoon/llm-data-analysis-course/blob/b1597e10c87f5c461bdb7e3f8f9b8454ba8cbedc/practice/chapter06/chapter06.md), [답안 양식](https://github.com/GilbertMoon/llm-data-analysis-course/blob/b1597e10c87f5c461bdb7e3f8f9b8454ba8cbedc/practice/chapter06/templates/chapter06_assignment.md)을 기준으로 했다. 공식 Notebook의 학습 순서와 추가 실습 6문항을 유지하고, 완료 주문 필터·안전한 병합·총합 검증·해석을 보완했다. eTL 상세 안내도 직접 확인했다. Chapter 05에서 저장한 processed CSV 4개를 사용하고, 질문·관찰·가설·추가 검증·판단을 Markdown에 기록한 개인 GitHub Notebook 파일 URL을 제출하는 조건에 맞췄다.

코드 작성·실행·독립 LLM 검토·문장 정리에 Codex의 도움을 받았다. LLM 응답과 실제 계산을 구분해 기록했으며, 핵심 실행 증거는 아래 저장된 코드 출력이다.


온라인 확인 기록(2026-10-08): eTL 상세 안내에 접근해 과제 요구사항과 마감을 확인했다. GitHub 게시·표시와 eTL 접수는 성공 여부를 따로 확인한다.


## 0-1. 실행 방법과 Evidence

저장소 루트에서 `python scripts/preprocess_data.py` 후 `python scripts/run_eda.py`를 실행한다. 또는 필요한 패키지를 설치한 Python 가상환경을 커널로 선택하고 이 Notebook을 위에서부터 전체 실행한다. 입력은 `data/processed/*_clean.csv`, 결과는 `reports/ch06_*`다. 원본 CSV는 수정하지 않는다.

이번에는 Notebook의 저장된 출력이 핵심 실행 Evidence다. 보고서 CSV 재읽기에서 빈 주석 문자열이 pandas 기본 설정으로 NaN이 되어 첫 대조가 실패했다. 원본 데이터 결측 처리와 구분하여, 보고서 CSV만 `keep_default_na=False`로 읽은 뒤 값 대조와 전체 재실행을 통과했다. 기존 수업 프로젝트의 가상환경(Python 3.14.3)을 재사용했다. 커널 경로는 아래 실제 출력에 남겼으며 이 제출 저장소에 새 `.venv`를 만들거나 VS Code 설정을 바꿨다고 주장하지 않는다. 마지막 셀에서 스크립트를 같은 Python 환경으로 실제 실행하고, CSV 재읽기 결과와 Notebook 집계를 대조한다.

## 1. EDA는 결론보다 질문에 가깝다

EDA는 Exploratory Data Analysis, 즉 탐색적 데이터 분석입니다. 이름 그대로 데이터를 여러 방향에서 탐색하면서 분포, 패턴, 차이, 관계, 이상한 값을 발견하는 과정입니다.

EDA에서 자주 던지는 질문은 다음과 같습니다.

- 데이터는 어떤 변수들로 구성되어 있는가?
- 주요 숫자형 변수의 분포는 어떤가?
- 주요 범주형 변수의 빈도는 어떤가?
- 특정 그룹 간 차이가 있는가?
- 시간에 따른 변화가 있는가?
- 이상하게 큰 값이나 작은 값이 있는가?
- 다음 단계에서 더 깊이 볼 질문은 무엇인가?


## 2. 관찰, 가설, 결론 구분하기

EDA 결과를 해석할 때는 관찰, 가설, 결론을 구분해야 합니다.

| 구분 | 의미 | 예시 |
|---|---|---|
| 관찰 | 데이터에서 직접 확인한 사실 | 전자기기 카테고리의 매출 비중이 가장 높다 |
| 가설 | 관찰을 바탕으로 생각해 볼 가능성 | 전자기기는 단가가 높아 매출 비중이 클 수 있다 |
| 결론 | 추가 검증 후 말할 수 있는 판단 | 단가와 판매 수량을 함께 분석한 결과 매출 차이의 주요 요인은 단가였다 |

이번 장에서는 주로 관찰과 가설을 만들고, 다음 분석 질문으로 연결하는 연습을 합니다.


## 3. 좋은 분석 질문의 조건

좋은 분석 질문은 데이터로 답할 수 있어야 합니다.

| 조건 | 설명 | 예시 |
|---|---|---|
| 데이터 기반 | 현재 데이터로 답할 수 있어야 함 | 월별 매출은 어떻게 변하는가? |
| 구체적 | 분석 대상과 기준이 명확해야 함 | 카테고리별 매출 비중은 어떻게 다른가? |
| 측정 가능 | 지표로 계산할 수 있어야 함 | 주문 수, 총매출, 평균 구매 금액 |
| 해석 가능 | 결과가 다음 판단과 연결되어야 함 | 어떤 상품군을 더 자세히 봐야 하는가? |
| 검증 가능 | 코드로 확인할 수 있어야 함 | `groupby()`로 계산 가능 |


## 4. 패키지와 경로 설정

저장소 루트와 chapter06 폴더 어느 쪽에서 실행해도 같은 입력을 찾는다. 실행 환경도 함께 출력했다.

In [1]:
from pathlib import Path
import sys
import hashlib
import subprocess
import pandas as pd
import numpy as np
from IPython.display import display

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = next((p for p in [CURRENT_DIR, *CURRENT_DIR.parents]
                     if (p / 'scripts' / 'run_eda.py').exists()
                     and (p / 'data' / 'processed').exists()), None)
assert PROJECT_ROOT is not None, '저장소 루트 아래에서 실행하세요.'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.eda import (load_processed_sales_data, run_basic_eda,
                     save_eda_outputs, build_eda_report)
print('Python:', sys.version.split()[0], 'pandas:', pd.__version__, 'NumPy:', np.__version__)
print('커널:', sys.executable)
print('입력:', PROCESSED_DIR.relative_to(PROJECT_ROOT))

Python: 3.14.3 pandas: 3.0.5 NumPy: 2.5.3
커널: /Users/sungmiiinn/Desktop/LLM을 활용한 빅데이터 분석/llm-data-analysis-course/.venv/bin/python
입력: data/processed


## 5. 전처리된 데이터 4개 불러오기 — Evidence E01

Chapter 02 피드백을 반영하여 customers만 읽지 않고 네 파일의 존재·크기·행열 수를 모두 확인했다.

In [2]:
data = load_processed_sales_data(PROCESSED_DIR)
customers, products, orders, order_items = [data[k].copy() for k in
    ['customers', 'products', 'orders', 'order_items']]
file_check = pd.DataFrame([{
    '파일': f'{name}_clean.csv', '존재': (PROCESSED_DIR / f'{name}_clean.csv').is_file(),
    '크기(bytes)': (PROCESSED_DIR / f'{name}_clean.csv').stat().st_size,
    '행': df.shape[0], '열': df.shape[1]
} for name, df in data.items()])
assert file_check['존재'].all() and file_check['크기(bytes)'].gt(0).all()
display(file_check)

,파일,존재,크기(bytes),행,열
0,customers_clean.csv,True,10358,150,11
1,products_clean.csv,True,4278,96,5
2,orders_clean.csv,True,21439,298,10
3,order_items_clean.csv,True,23047,724,7


## 6. 데이터 크기와 컬럼 확인하기

EDA를 시작하기 전에 각 데이터의 행/열 수와 컬럼명을 다시 확인합니다. CSV로 저장했다가 다시 불러오면 날짜 컬럼이 문자열로 돌아올 수 있으므로 날짜형 변환도 다시 확인합니다.


In [3]:
structure = pd.concat([pd.DataFrame({'데이터':name,'컬럼':df.columns,
    'dtype':df.dtypes.astype(str).to_numpy(), '결측수':df.isna().sum().to_numpy()})
    for name,df in data.items()], ignore_index=True)
display(structure)

,데이터,컬럼,dtype,결측수
0,customers,customer_id,int64,0
1,customers,name,str,0
2,customers,gender,str,0
3,customers,age,float64,0
4,customers,city,str,0
5,customers,signup_date,str,2
6,customers,age_imputed,bool,0
7,customers,age_outlier_candidate,bool,0
8,customers,city_imputed,bool,0
9,customers,signup_date_invalid,bool,0


In [4]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')
orders['order_month'] = orders['order_date'].dt.strftime('%Y-%m')
print('재로딩 후 order_date 결측:', int(orders.order_date.isna().sum()))
print('clean 전체 주문일:', orders.order_date.min(), '~', orders.order_date.max())
results = run_basic_eda(data)
parameters = dict(results['parameters'].values)
display(results['scope'])
display(results['parameters'])

재로딩 후 order_date 결측: 3
clean 전체 주문일: 2025-07-01 00:00:00 ~ 2028-12-31 00:00:00


,범위,상세행수,주문수,금액합계,주석
0,전체 clean 상세 (매출 아님),724,297,244374000,
1,completed 전체,430,178,137335000,
2,completed 날짜 결측 제외,6,2,1903000,금액분석의 공통 제외
3,completed 미래 후보 제외,3,1,909000,기준일 이후를 공통 제외
4,최종 분석 모집단,421,175,134523000,금액 상세가 없는 조건충족 주문 1건; 평균 분모 제외


,항목,값
0,기준일,2026-10-08
1,기간 시작,2025-07-02
2,기간 종료,2026-08-28
3,고액 1회 기준(Q75 원),1098500.0
4,상세 없는 조건충족 주문 수,1
5,전체 부분보존 주문 수,31
6,분석 내 부분보존 주문 수,18


## 7. 내가 정한 EDA 질문 — Evidence E02

아래 질문은 기간, 상태, 금액 기준, 필요한 테이블, 지표, 검증 방법을 함께 정했다. 주문 수는 상세 행 수가 아니라 `order_id.nunique()`다. 이 컬럼들이 실제로 존재하고 FK 연결을 통과했기 때문에 계산할 수 있다. 고객수가 많은 곳과 구매금액이 큰 곳이 같은지, 금액 차이가 수량·단가·건수 중 어디에서 나타나는지가 다음 분석에 도움이 된다고 보았다.

In [5]:
questions = results['questions']
display(questions)

,질문,분석 범위,필요 데이터,지표,계산 방법,검증 방법
0,카테고리별 금액 차이는 수량과 거래 단가에서 어떻게 나타나는가?,"2025-07-02~2026-08-28 양 끝 포함, completed, 유효 주문...","orders, order_items, products",금액·수량·수량가중 거래단가,groupby sum; 금액/수량,카테고리 합계=집계 전 clean completed 합계
1,월별 주문 수와 평균 주문금액은 어떻게 달라지는가?,"2025-07-02~2026-08-28 양 끝 포함, completed, 유효 주문...","orders, order_items",월 금액·고유 주문 수·주문당 금액,order_id별 합산 후 월별 집계,월 합계=집계 전 clean; 두 집계 방식 일치
2,도시별 고객 수와 완료 주문금액 순위는 같은가?,"2025-07-02~2026-08-28 양 끝 포함, completed, 유효 주문...","customers, orders, order_items",도시 고객수·구매고객수·금액·고객당 금액,customer_id→city many_to_one,도시 합계=집계 전 clean; 고객수 합계=clean 고객수
3,결제수단별 평균 주문금액은 어떻게 다른가?,"2025-07-02~2026-08-28 양 끝 포함, completed, 유효 주문...","orders, order_items",결제수단별 주문금액 평균·주문 수,라인→주문 합산→결제수단별 mean,주문금액 총합=집계 전 clean; 라인평균과 구분
4,반복 구매 고객과 고액 1회 고객은 몇 명인가?,"2025-07-02~2026-08-28 양 끝 포함, completed, 유효 주문...","customers, orders, order_items",고유 주문수·총금액·평균 주문금액,고객별 nunique; 반복≥2; 고액1회≥전체 주문금액75%분위,고객 합계=집계 전 clean; 분류 기준·기간 고정


## 8. 고객 데이터 EDA

먼저 고객 데이터만 따로 살펴봅니다. 하나의 데이터셋 안에서 분포와 빈도를 확인하면 이후 병합 분석을 할 때 기준이 생깁니다.


In [6]:
display(customers['age'].describe().to_frame('clean_age'))
print('18~100세 밖 후보:', int((~customers.age.between(18,100)).sum()))
print('age_imputed:', int(customers.age_imputed.sum()) if 'age_imputed' in customers else '미제공')

,clean_age
count,150.000000
mean,43.913333
std,17.802397
min,18.000000
25%,30.000000
50%,41.000000
75%,56.750000
max,150.000000


18~100세 밖 후보: 1
age_imputed: 6


In [7]:
customer_city = results['customer_city']
display(customer_city)

,city,customer_count,ratio_pct
0,부산,23,15.33
1,대구,21,14.00
2,광주,21,14.00
3,서울,18,12.00
4,울산,17,11.33
5,인천,16,10.67
6,수원,10,6.67
7,성남,9,6.00
8,고양,8,5.33
9,대전,5,3.33


In [8]:
customer_gender = results['customer_gender']
display(customer_gender)

,gender,customer_count,ratio_pct
0,M,80,53.33
1,F,70,46.67


### 결과 관찰

clean 고객은 150명이다. 고객수가 가장 많은 도시는 부산(23명)다. 나이 이상 후보와 보정 여부는 바로 위 출력에 함께 남겼다.

### 나의 해석과 판단

등록 고객수가 많다고 구매금액도 클 것이라고 바로 결론 내리지 않았다. 아래 추가 실습에서 실제 구매 고객과 금액을 연결해 비교했다. 나이 평균은 대체값과 150세 후보의 영향을 받으므로 그 자체로 고객 특성을 단정하기 어렵다.

### 업무·분석적 의미

도시별 고객수와 구매 고객수를 나누면 고객 기반이 큰 곳과 기간 내 거래가 활발한 곳을 구분할 수 있다.

### 한계와 추가 확인 사항

`Unknown`은 결측을 묶은 값이다. 실제 지역으로 해석하지 않는다. 합성 데이터이며 전처리에서 대체한 나이는 원래 관측된 값과 다르다.

## 9. 상품 데이터 EDA

상품 데이터에서는 카테고리와 가격을 먼저 봅니다. 상품 수가 많은 카테고리가 매출도 높은지는 아직 알 수 없습니다. 그것은 주문 상세와 연결한 뒤 확인해야 합니다.


In [9]:
product_category = results['product_category']
display(product_category)

,category,product_count,ratio_pct
0,뷰티,17,17.71
1,패션,16,16.67
2,식품,16,16.67
3,전자기기,14,14.58
4,도서,13,13.54
5,스포츠,10,10.42
6,생활용품,10,10.42


In [10]:
display(products['price'].describe().to_frame('catalog_price'))
display(results['price_check'])

,catalog_price
count,9.600000e+01
mean,2.048958e+05
std,1.011848e+06
min,5.000000e+03
25%,5.275000e+04
50%,9.450000e+04
75%,1.532500e+05
max,9.999999e+06


,확인,건수,최댓값
0,상품 목록 가격 후보,1,9999999
1,해당 상품의 completed 상세,3,187000


In [11]:
category_price = results['category_price']
display(category_price)

,category,product_count,avg_price,min_price,max_price
0,도서,13,103692.307692,34000,182000
1,뷰티,17,672294.058824,18000,9999999
2,생활용품,10,75000.000000,29000,147000
3,스포츠,10,78400.000000,5000,183000
4,식품,16,120812.500000,11000,195000
5,전자기기,14,113142.857143,21000,178000
6,패션,16,115125.000000,28000,197000


## 10. 주문 데이터 EDA

목록 가격에는 이상 후보가 남아 있으므로 카테고리 목록 평균가격을 거래 단가의 대용으로 쓰지 않았다. 실제 거래금액은 주문 당시 `unit_price`를 사용한다. 주문 상태·결제수단 분포는 clean 주문 전체 기준으로 먼저 보고, 금액성 분석은 별도의 completed 공통 범위로 진행한다.

In [12]:
order_status = results['order_status']
display(order_status)

,order_status,order_count,ratio_pct
0,completed,179,60.07
1,cancelled,62,20.81
2,refunded,57,19.13


In [13]:
payment_method = results['payment_method']
display(payment_method)

,payment_method,order_count,ratio_pct
0,kakao_pay,89,29.87
1,card,72,24.16
2,bank_transfer,70,23.49
3,naver_pay,67,22.48


In [14]:
print('금액 분석 시작일:', parameters['기간 시작'])
print('금액 분석 종료일:', parameters['기간 종료'])
print('기준일: 2026-10-08; 완료상태: completed')
print('서로 다른 분석 월:', results['monthly_sales'].order_month.nunique())

금액 분석 시작일: 2025-07-02
금액 분석 종료일: 2026-08-28
기준일: 2026-10-08; 완료상태: completed
서로 다른 분석 월: 14


## 11. 금액 분석을 위한 데이터 연결

공식 예제는 상품만 연결하면 cancelled/refunded까지 금액에 섞일 수 있었다. 먼저 orders를 연결하고 completed·유효 날짜·기준일 조건을 적용한 뒤 products와 customers를 연결했다. 세 병합 모두 `validate="many_to_one"`과 미연결·행 증가 검사를 사용했다. 세부 구현은 `src/eda.py`의 `run_basic_eda()`에 있다.

In [15]:
calculated_line_total = order_items.quantity * order_items.unit_price
assert order_items.line_total.equals(calculated_line_total)
print('line_total 독립 재계산 불일치:', int(order_items.line_total.ne(calculated_line_total).sum()))
display(order_items[['quantity','unit_price','line_total']].head())

line_total 독립 재계산 불일치: 0


,quantity,unit_price,line_total
0,2,172000,344000
1,4,79000,316000
2,3,87000,261000
3,1,187000,187000
4,5,125000,625000


In [16]:
sales_items = results['sales_base'].copy()
print('최종 completed 상세 행:', len(sales_items))
print('고유 주문 수:', sales_items.order_id.nunique())
display(results['validation'].iloc[:6])
display(sales_items[['order_item_id','order_id','category','quantity','unit_price','line_total','order_date','order_status']].head())

최종 completed 상세 행: 421
고유 주문 수: 175


,검증,실제값,기대값,통과
0,FK customer_id,0,0,True
1,FK order_id,0,0,True
2,FK product_id,0,0,True
3,orders 병합 행 증가,0,0,True
4,products 병합 행 증가,0,0,True
5,customers 병합 행 증가,0,0,True


,order_item_id,order_id,category,quantity,unit_price,line_total,order_date,order_status
0,1,1,뷰티,2,172000,344000,2026-07-12,completed
1,2,2,뷰티,4,79000,316000,2025-10-02,completed
2,3,2,생활용품,3,87000,261000,2025-10-02,completed
3,4,2,패션,1,187000,187000,2025-10-02,completed
4,5,2,전자기기,5,125000,625000,2025-10-02,completed


## 12. 카테고리별 매출 EDA — Evidence E03

카테고리별 매출은 어떤 상품군이 매출에 많이 기여했는지 보여 줍니다. 다만 매출이 높은 이유가 판매 수량 때문인지, 단가 때문인지는 추가로 확인해야 합니다.


In [17]:
category_sales = sales_items.groupby('category', as_index=False).agg(
    total_quantity=('quantity','sum'), total_sales=('line_total','sum'),
    order_count=('order_id','nunique'), sold_product_count=('product_id','nunique')
).sort_values('total_sales', ascending=False)
category_sales['sales_ratio_pct'] = (category_sales.total_sales / category_sales.total_sales.sum()*100).round(2)
category_sales['quantity_weighted_unit_price'] = category_sales.total_sales/category_sales.total_quantity
pd.testing.assert_frame_equal(category_sales.reset_index(drop=True), results['category_sales'].reset_index(drop=True))
display(category_sales)

,category,total_quantity,total_sales,order_count,sold_product_count,sales_ratio_pct,quantity_weighted_unit_price
6,패션,290,32940000,68,15,24.49,113586.206897
4,식품,220,24731000,61,16,18.38,112413.636364
5,전자기기,172,20445000,53,14,15.20,118866.279070
1,뷰티,176,17166000,53,17,12.76,97534.090909
0,도서,168,16895000,47,13,12.56,100565.476190
3,스포츠,165,13237000,47,10,9.84,80224.242424
2,생활용품,125,9109000,42,10,6.77,72872.000000


In [18]:
category_sales_with_price = category_sales.merge(category_price[['category','avg_price']], on='category', validate='one_to_one')
display(category_sales_with_price)
print('avg_price=목록 평균가격; quantity_weighted_unit_price=실제 거래금액/판매수량. 서로 다른 지표다.')

,category,total_quantity,total_sales,order_count,sold_product_count,sales_ratio_pct,quantity_weighted_unit_price,avg_price
0,패션,290,32940000,68,15,24.49,113586.206897,115125.000000
1,식품,220,24731000,61,16,18.38,112413.636364,120812.500000
2,전자기기,172,20445000,53,14,15.20,118866.279070,113142.857143
3,뷰티,176,17166000,53,17,12.76,97534.090909,672294.058824
4,도서,168,16895000,47,13,12.56,100565.476190,103692.307692
5,스포츠,165,13237000,47,10,9.84,80224.242424,78400.000000
6,생활용품,125,9109000,42,10,6.77,72872.000000,75000.000000


avg_price=목록 평균가격; quantity_weighted_unit_price=실제 거래금액/판매수량. 서로 다른 지표다.


### 결과 관찰

패션의 completed 금액이 32,940,000원으로 가장 크다. 수량은 290개이며 수량가중 거래단가는 113,586원이다. 판매수량 1위는 패션(290개)다.

### 나의 해석과 판단

금액 순위만으로 선호도를 말하지 않았다. 금액은 수량과 거래 단가의 곱이므로 두 지표를 같이 봤다. 위 표에서 수량 1위와 금액 1위가 같은지 직접 대조할 수 있다.

### 업무·분석적 의미

다음 분석에서는 금액이 큰 카테고리의 판매량과 거래가격 구성을 나누어 확인하는 것이 유용하다.

### 한계와 추가 확인 사항

원가·할인·배송비·수수료가 없어 이익과 순매출을 계산하지 못한다. 목록 평균가격의 극단 후보는 실제 거래 단가를 바꾸는 근거가 아니다.

## 13. 월별 매출과 주문 수 EDA

월별 매출을 보려면 주문 상세와 주문 정보를 `order_id` 기준으로 연결합니다. 월별 매출은 시간 흐름을 보여 주지만, 원인을 바로 설명하지는 않습니다.


In [19]:
order_sales = sales_items.copy()
order_amounts = results['order_amounts'].copy()
print('상세 행:',len(order_sales),' / 금액 있는 completed 주문:',len(order_amounts))
print('분석 주문일 결측:',int(order_sales.order_date.isna().sum()))
display(order_amounts[['order_id','order_month','payment_method','order_total','item_rows']].head())

상세 행: 421  / 금액 있는 completed 주문: 175
분석 주문일 결측: 0


,order_id,order_month,payment_method,order_total,item_rows
0,1,2026-07,naver_pay,344000,1
1,2,2025-10,kakao_pay,1389000,4
2,3,2026-03,naver_pay,418000,3
3,4,2025-10,kakao_pay,116000,1
4,6,2025-07,bank_transfer,2082000,3


In [20]:
monthly_sales = order_amounts.groupby('order_month',as_index=False).agg(
    total_sales=('order_total','sum'), order_count=('order_id','nunique'),
    avg_order_value=('order_total','mean')).sort_values('order_month')
pd.testing.assert_frame_equal(monthly_sales.reset_index(drop=True),results['monthly_sales'].reset_index(drop=True))
display(monthly_sales)

,order_month,total_sales,order_count,avg_order_value
0,2025-07,9479000,11,8.617273e+05
1,2025-08,8892000,17,5.230588e+05
2,2025-09,4519000,4,1.129750e+06
3,2025-10,12547000,17,7.380588e+05
4,2025-11,9163000,16,5.726875e+05
5,2025-12,8113000,12,6.760833e+05
6,2026-01,7759000,8,9.698750e+05
7,2026-02,7689000,12,6.407500e+05
8,2026-03,7493000,13,5.763846e+05
9,2026-04,11459000,14,8.185000e+05


### 결과 관찰

분석 기간은 2025-07-02~2026-08-28이며 월 14개가 포함된다. 실제 거래일의 최소·최대로 분석 기간을 정했고, 경계 월은 보수적으로 제외하여 중간 월만 비교했다. 데이터의 실제 추출 시작·종료일은 없어 경계 월 전체가 수집되었는지는 확인할 수 없다. 그중 금액 최대인 2025-10은 12,547,000원·17건·주문당 738,059원이고, 최소인 2025-09은 4,519,000원·4건·주문당 1,129,750원이다.

### 나의 해석과 판단

전체 금액 차이는 주문 건수와 주문당 금액을 함께 봐야 한다. 같은 월이라도 실제 관측일수가 다를 수 있어 처음·마지막 월을 동일한 관측 범위라고 가정하여 비교하지 않았다.

### 업무·분석적 의미

월별 변화를 조사할 때 먼저 건수와 객단가, 그다음 월×카테고리 구성을 확인하는 순서가 적절하다고 판단했다.

### 한계와 추가 확인 사항

표의 차이는 기술적 비교다. 광고·계절성·고객 이탈이 원인이라고 결론 내릴 수 없으며, 한 해 안팎의 샘플로 계절성을 확정할 수 없다.

## 14. 고객별 구매 금액 EDA

고객별 구매 금액은 우수 고객 분석의 출발점입니다. 하지만 한 번의 고액 구매 고객과 여러 번 반복 구매한 고객은 다르게 해석해야 합니다. 그래서 `total_sales`, `order_count`, `avg_order_value`를 함께 봅니다.


In [21]:
customer_sales_base = sales_items.copy()
customer_sales = results['customer_sales']
display(customer_sales.head(10))
print('구매 고객:', len(customer_sales), ' / clean 전체 고객:', len(customers))

,customer_id,city,order_count,total_sales,avg_order_value,customer_group
1,3,서울,3,6702000,2.234000e+06,반복 구매 (2회 이상)
7,14,대구,4,5122000,1.280500e+06,반복 구매 (2회 이상)
56,90,수원,3,4311000,1.437000e+06,반복 구매 (2회 이상)
46,65,광주,6,3977000,6.628333e+05,반복 구매 (2회 이상)
65,104,서울,2,3966000,1.983000e+06,반복 구매 (2회 이상)
58,93,울산,4,3845000,9.612500e+05,반복 구매 (2회 이상)
11,20,성남,4,3808000,9.520000e+05,반복 구매 (2회 이상)
27,38,성남,4,3656000,9.140000e+05,반복 구매 (2회 이상)
19,28,인천,4,3201000,8.002500e+05,반복 구매 (2회 이상)
54,85,부산,4,2832000,7.080000e+05,반복 구매 (2회 이상)


구매 고객: 96  / clean 전체 고객: 150


## 15. EDA 결과를 다음 질문으로 연결하기

EDA의 가치는 표를 만드는 데서 끝나지 않습니다. 각 결과가 다음 질문으로 이어져야 합니다.


In [22]:
eda_result_summary = results['eda_result_summary']
display(eda_result_summary)

,관찰 주제,결과표,다음 질문
0,도시 고객수와 금액 차이,city_sales,구매 고객 비중·주문 수·고객당 금액을 함께 비교
1,상품 수와 거래금액 차이,category_comparison,판매 수량과 수량가중 거래단가를 함께 비교
2,월별 차이,"monthly_sales, month_category",경계 월을 제외하고 주문수와 객단가를 분리
3,고액 고객의 반복성,customer_groups,기간 내 2회 이상과 고액 1회를 분리; 미래 재구매는 미확인


## 16. EDA 결과 저장하기

EDA 결과표를 CSV로 저장하면 다음 장의 시각화나 보고서 작성에서 다시 사용할 수 있습니다.


In [23]:
saved_paths = save_eda_outputs(results, REPORT_DIR)
reloaded = []
for name, frame in results.items():
    if name in {'sales_base','order_amounts'}:
        continue
    saved = pd.read_csv(REPORT_DIR / f'ch06_{name}.csv', keep_default_na=False)
    # CSV 문자열 추론 차이를 허용하되 값·열·행 순서는 그대로 대조한다.
    pd.testing.assert_frame_equal(saved, frame.reset_index(drop=True), check_dtype=False)
    reloaded.append({'파일': f'ch06_{name}.csv', '행':len(saved), '재읽기 일치':True})
display(pd.DataFrame(reloaded))

,파일,행,재읽기 일치
0,ch06_questions.csv,5,True
1,ch06_customer_city.csv,11,True
2,ch06_customer_gender.csv,2,True
3,ch06_product_category.csv,7,True
4,ch06_category_price.csv,7,True
5,ch06_order_status.csv,3,True
6,ch06_payment_method.csv,4,True
7,ch06_category_sales.csv,7,True
8,ch06_monthly_sales.csv,14,True
9,ch06_customer_sales.csv,96,True


## 17. EDA 요약 보고서 만들기

간단한 Markdown 요약 보고서를 만들어 봅니다. 보고서에는 분석 목적, 주요 질문, 핵심 결과, 추가 질문, 해석 시 주의사항을 담습니다.


In [24]:
report_path = REPORT_DIR/'ch06_eda_summary.md'
report_path.write_text(build_eda_report(results), encoding='utf-8')
print('저장:', report_path.relative_to(PROJECT_ROOT), ' / bytes:', report_path.stat().st_size)

저장: reports/ch06_eda_summary.md  / bytes: 8716


## 18. 소스 모듈로 같은 EDA 실행하기

위에서 노트북으로 한 단계씩 실행한 EDA는 `src/eda.py`에 함수로 정리되어 있습니다. 반복 실행하거나 프로젝트 코드로 관리하려면 노트북보다 소스 모듈을 사용하는 것이 좋습니다.


In [25]:
module_data = load_processed_sales_data(PROCESSED_DIR)
module_results = run_basic_eda(module_data)
for key in ['category_sales','monthly_sales','customer_sales','validation']:
    pd.testing.assert_frame_equal(results[key],module_results[key])
print('Notebook 집계와 모듈 재실행 4개 핵심 표 일치')

Notebook 집계와 모듈 재실행 4개 핵심 표 일치


In [26]:
display(module_results['validation'])
assert module_results['validation']['통과'].all()
print('모든 검증 통과')

,검증,실제값,기대값,통과
0,FK customer_id,0,0,True
1,FK order_id,0,0,True
2,FK product_id,0,0,True
3,orders 병합 행 증가,0,0,True
4,products 병합 행 증가,0,0,True
5,customers 병합 행 증가,0,0,True
6,집계 전 clean 독립 계산 합계,134523000,134523000,True
7,병합 상세 합계,134523000,134523000,True
8,카테고리 합계,134523000,134523000,True
9,월 합계,134523000,134523000,True


모든 검증 통과


## 19. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 EDA 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/run_eda.py
```

이 스크립트는 6장 EDA 결과 CSV와 `reports/ch06_eda_summary.md`를 자동으로 저장합니다.


## 20. 실제 LLM 질문 확장 기록 — Evidence E05

Prompt에는 아래 구조와 분석 제약만 전달했다. 별도 검토 에이전트는 수업용 합성 원본의 일부 예시 행도 로컬에서 확인했다. 2026-10-08 Codex의 별도 검토 에이전트가 실제로 답했고, 응답을 아래에 기록했다. 교재의 가상 매출 예시를 이번 데이터 결과로 복사하지 않았다.

**Safe Context**: 수업용 합성 데이터. customers(customer_id, name, gender, age, city, signup_date), products(product_id, product_name, category, price), orders(order_id, customer_id, order_date, payment_method, order_status), order_items(order_item_id, order_id, product_id, quantity, unit_price, line_total). clean의 PK/FK 확인. 모든 금액은 completed·유효 과거 주문일·고정 기준일, quantity×unit_price. 광고·유입·원가·재고 컬럼 없음.

**실제 Prompt**

```text
(1) 현재 컬럼으로 검증 가능한 다음 EDA 질문 5개를 필요한 테이블/지표/pandas기능과 함께 제안해주세요. 인과결론은 내리지 마세요.
(2) 답할 수 없는 '광고 때문에 매출이 증가했나?'를 현재 데이터로 가능한 질문으로 고쳐주세요.
(3) 제안의 사용/수정/보류 판단에 필요한 한계를 짚어주세요.
```

**실제 응답 요약**

1. 카테고리별 수량·거래금액·비중을 merge와 groupby로 비교하고 총합 검증.
2. 월×카테고리 금액, 주문 수, 주문당 금액을 비교. 부분 관측 월 주의.
3. 도시별 전체 고객수·구매 고객수·금액을 비교. Unknown은 실제 도시로 해석 금지.
4. 주문별로 먼저 합산한 결제수단별 평균·중앙값 비교. 결제수단 효과라는 인과 해석 금지.
5. 기간 내 2회 이상 반복 구매, 1회이며 고객 총금액 Q90 이상인 고액 단건 고객 분류. 기준값·기간·민감도 기록.
6. 광고 인과 질문을 월별 completed 금액·주문 수·주문당 금액·카테고리 구성의 동반 변화 질문으로 수정. 광고의 인과효과·이익·재고 회전율은 데이터가 없어 보류.

응답의 원문은 같은 폴더의 [LLM 검토 기록](llm_review.md)에 남겼다.

## 21. LLM 제안과 실제 데이터의 대조

| 제안 | 판단 | 실제 확인과 수정 이유 |
| --- | --- | --- |
| 카테고리 수량·금액 | 사용 | 실제 orders/order_items/products 연결과 총합 검증 통과. 금액 134,523,000원 |
| 월×카테고리 변화 | 사용 | 아래 실습 4의 표를 실제 생성하고 전체 합계를 대조. 경계 월은 설명에 별도 표시 |
| 도시 고객수·금액 | 사용 | 아래 실습 1에서 clean 전체 고객수와 구매고객수를 구분해 계산. Unknown 유지 |
| 결제수단 평균·중앙값 | 사용 | 아래 실습 3에서 주문당 합계로 먼저 줄인 후 평균·중앙값 계산. 라인 평균도 나란히 출력 |
| 고액 1회 고객 Q90 | 수정 후 사용 | 반복 기준 2회는 유지. 고액 기준을 **전체 분석 주문금액 Q75(1,098,500원)**로 정해 주문 단위를 맞췄다. LLM의 구매고객 총금액 Q90도 아래에서 계산해 민감도 대조 |
| 광고 때문에 금액이 늘었는가 | 보류 → 기술적 질문으로 수정 | 광고 자료가 없어서 인과효과는 보류. 월별 금액·건수·객단가와 월×카테고리 변화를 계산 |
| 이익·재고 회전율 | 보류 | 원가·재고 컬럼 없음. 거래금액을 이익으로 부르지 않음 |

**답안에서 수정한 내용**: 필터를 모든 금액표에 공통 적용했고, 주문 수를 `nunique(order_id)`로, 평균은 라인 평균이 아닌 주문 합계의 평균으로 고쳤다. 이 기준 작성과 재검증에도 Codex의 도움을 받았다. 실제 출력으로 검증한 사실과 설명을 위한 가설을 구분했다.

## 22. 추가 실습 6문항

아래는 공식 Notebook의 추가 문항에 대한 답이다. 모든 구매금액은 앞서 확정한 같은 completed 모집단이며 고객·상품 전체 개수와 구매 개수의 분모를 구분했다.

In [27]:
# 1. 도시별 고객 수와 총 구매금액 비교
city_sales = results['city_sales']
display(city_sales)
assert city_sales.total_sales.sum() == sales_items.line_total.sum()
print('도시 고객수 합계:', int(city_sales.customer_count.sum()))

,city,customer_count,ratio_pct,total_sales,order_count,buyer_count,sales_per_registered_customer,customer_rank,sales_rank
0,부산,23,15.33,22440000,30,15,9.756522e+05,1,1
3,서울,18,12.00,21479000,20,11,1.193278e+06,4,2
2,광주,21,14.00,19715000,28,16,9.388095e+05,2,3
5,인천,16,10.67,15723000,23,12,9.826875e+05,6,4
1,대구,21,14.00,13811000,18,12,6.576667e+05,2,5
6,수원,10,6.67,12220000,13,8,1.222000e+06,7,6
7,성남,9,6.00,11288000,13,5,1.254222e+06,8,7
4,울산,17,11.33,10213000,16,8,6.007647e+05,5,8
8,고양,8,5.33,6442000,10,6,8.052500e+05,9,9
10,Unknown,2,1.33,820000,1,1,4.100000e+05,11,10


도시 고객수 합계: 150


In [28]:
# 2. 상품 수가 많은 카테고리와 금액 순위 비교
category_comparison = results['category_comparison']
display(category_comparison[['category','product_count','sold_product_count','total_quantity','total_sales','quantity_weighted_unit_price','product_rank','sales_rank']])
print('상품수 합계:', int(category_comparison.product_count.sum()))

,category,product_count,sold_product_count,total_quantity,total_sales,quantity_weighted_unit_price,product_rank,sales_rank
1,패션,16,15,290,32940000,113586.206897,2,1
2,식품,16,16,220,24731000,112413.636364,2,2
3,전자기기,14,14,172,20445000,118866.279070,4,3
0,뷰티,17,17,176,17166000,97534.090909,1,4
4,도서,13,13,168,16895000,100565.476190,5,5
5,스포츠,10,10,165,13237000,80224.242424,6,6
6,생활용품,10,10,125,9109000,72872.000000,6,7


상품수 합계: 96


In [29]:
# 3. 결제수단별 주문당 평균·중앙값; 라인 평균은 별도 지표
payment_order = results['payment_order']
display(payment_order)
assert payment_order.order_count.sum() == order_amounts.order_id.nunique()
assert payment_order.total_sales.sum() == sales_items.line_total.sum()

,payment_method,order_count,total_sales,avg_order_value,median_order_value,line_count,mean_line_total
0,card,40,37903000,947575.000000,693000.0,110,344572.727273
1,bank_transfer,48,35748000,744750.000000,715000.0,112,319178.571429
2,kakao_pay,49,35854000,731714.285714,608000.0,118,303847.457627
3,naver_pay,38,25018000,658368.421053,500500.0,81,308864.197531


In [30]:
# 4. 월×카테고리 금액표
month_category = results['month_category']
month_category_pivot = month_category.pivot_table(index='order_month',columns='category',values='total_sales',aggfunc='sum',fill_value=0)
display(month_category_pivot)
assert month_category_pivot.to_numpy().sum() == sales_items.line_total.sum()
print('월×카테고리 총합:', int(month_category_pivot.to_numpy().sum()))

category,도서,뷰티,생활용품,스포츠,식품,전자기기,패션
order_month,,,,,,,
2025-07,1040000,1133000,406000,295000,2123000,1319000,3163000
2025-08,1144000,1050000,983000,417000,505000,2547000,2246000
2025-09,380000,125000,735000,438000,528000,1157000,1156000
2025-10,1570000,661000,1334000,1426000,3088000,1781000,2687000
2025-11,583000,1581000,528000,1271000,2221000,1035000,1944000
2025-12,34000,884000,489000,791000,2048000,1408000,2459000
2026-01,74000,0,804000,2359000,2465000,590000,1467000
2026-02,1827000,275000,646000,1009000,1193000,1562000,1177000
2026-03,1075000,728000,233000,364000,3178000,1378000,537000


월×카테고리 총합: 134523000


### 추가 실습 1~4의 관찰과 판단

- **관찰**: 고객수 1위는 부산(23명), 금액 1위는 부산(22,440,000원)다. 상품수 1위는 뷰티(17개), 금액 1위는 패션이다. 결제수단별 평균·중앙값은 바로 위 실습 3의 주문 단위 계산값이며 라인 평균과 구별했다.
- **나의 해석과 판단**: 개수와 거래금액의 순위는 같은 지표가 아니다. 도시마다 고객수뿐 아니라 구매고객수와 주문수가 다를 수 있고, 카테고리는 판매량·거래단가가 함께 작용한다.
- **업무·분석적 의미**: 지역과 상품 구성의 차이를 다음 비교 항목으로 좁힐 수 있다. 결제수단은 단위가 맞는 평균 주문금액부터 확인해야 한다.
- **한계와 추가 확인 사항**: 이 차이를 지역 성향·상품 선호도·결제수단의 효과로 단정하지 않는다. 상품별 노출·광고·재고와 관측기간 밖 고객 행동이 없다.

### 일부 상세가 제외된 주문의 영향

원본 중복 제거 후 고유 상세 수와 clean 상세 수를 주문별로 비교했다. 전체 31건은 일부 상세만 남았고, 현재 분석 안에는 18건이 있다. 기본 결과는 남은 유효 상세를 포함하며, 아래에서 이 18건을 전부 제외한 결과도 비교했다. 조건은 같고 부분 보존 주문의 포함 여부만 다르다. 이는 원래 청구금액을 복원한 값이 아니라 결과의 민감도를 확인한 것이다. 상세가 아예 없는 조건충족 주문 1건은 평균의 분모에서 제외했다.

In [31]:
display(results['partial_order_audit'].loc[results['partial_order_audit'].in_analysis])
display(results['partial_sensitivity'])
assert int(parameters['분석 내 부분보존 주문 수']) == 18

,order_id,raw_unique_item_count,clean_item_count,excluded_item_count,partially_retained,in_analysis
8,9,4,3.0,1.0,True,True
35,36,4,3.0,1.0,True,True
46,47,3,2.0,1.0,True,True
85,86,3,2.0,1.0,True,True
104,105,4,3.0,1.0,True,True
110,111,3,2.0,1.0,True,True
128,129,4,3.0,1.0,True,True
129,130,2,1.0,1.0,True,True
163,164,4,3.0,1.0,True,True
178,179,4,3.0,1.0,True,True


,분석 범위,상세행수,주문수,금액합계,남은상세 주문금액평균
0,남은 유효 상세 포함 (기본),421,175,134523000,768702.857143
1,일부 상세 제외 주문 전부 제외,377,157,120972000,770522.292994


### 추가 실습 5. 반복 구매와 고액 1회 구매의 기준

In [32]:
# 반복: 같은 기간의 금액 있는 completed 주문이 2건 이상.
# 고액 1회: 1건이며 주문금액이 전체 분석 주문금액 Q75 이상.
q75 = order_amounts.order_total.quantile(.75)
q90 = customer_sales.total_sales.quantile(.9)
display(results['customer_groups'])
threshold_sensitivity = pd.DataFrame([
    ['주문금액 Q75 (이번 기준)',q75,int((customer_sales.order_count.eq(1)&customer_sales.avg_order_value.ge(q75)).sum())],
    ['고객 총금액 Q90 (LLM 제안)',q90,int((customer_sales.order_count.eq(1)&customer_sales.total_sales.ge(q90)).sum())]
],columns=['기준','기준값(원)','고액1회 고객수'])
display(threshold_sensitivity)
assert results['customer_groups'].customer_count.sum()==len(customer_sales)
assert results['customer_groups'].total_sales.sum()==sales_items.line_total.sum()

,customer_group,customer_count,order_count,total_sales,mean_customer_sales
2,반복 구매 (2회 이상),53,132,105215000,1.985189e+06
1,기타 1회 구매,36,36,19122000,5.311667e+05
0,고액 1회 구매 (주문금액 Q75 이상),7,7,10186000,1.455143e+06


,기준,기준값(원),고액1회 고객수
0,주문금액 Q75 (이번 기준),1098500.0,7
1,고객 총금액 Q90 (LLM 제안),2789500.0,0


### 추가 실습 5의 관찰과 판단

**관찰**: 고객 유형별 인원·주문수·금액과 두 고액 기준의 결과는 위 출력과 같다. 한 고객의 주문 상세가 여러 행이어도 고유 주문 수만 센다.

**나의 해석과 판단**: 반복 주문과 한 번의 큰 거래를 분리할 수는 있다. 다만 고액 구분은 샘플 안의 비교 기준이어서 고객의 평생가치나 충성도를 뜻하지 않는다. Q75와 Q90의 모집단도 각각 주문·구매고객으로 다르다고 명시했다.

**업무·분석적 의미**: 재구매 고객과 고액 단건 고객을 다른 후속 질문으로 검토하는 출발점이다.

**한계와 추가 확인 사항**: 관측기간 이전·이후의 주문은 모르며 고객 유지율과 미래 매출 예측은 이번 분석 범위가 아니다.

### 추가 실습 6. 답할 수 없는 질문을 수정하기

20번에서 실제 요청한 Prompt의 (2)가 이 문항에 해당한다.

- 원래 질문: **광고 때문에 매출이 증가했나?**
- LLM의 실제 수정: **관측 기간에 월별 completed 거래금액은 어떻게 변했고, 주문 수·주문당 금액·카테고리 구성은 함께 어떻게 달라졌나?**
- 판단: **수정 후 사용**. 광고 인과효과는 보류하고 월별 표, 주문당 금액, 월×카테고리 표로 답했다. 필요한 데이터가 현재 네 테이블에 있고 총합 검증도 통과했다.
- 남은 한계: 이 수정 질문에 답해도 광고 효과를 증명한 것은 아니다. 광고비·노출·유입·시점·비교집단이 추가로 필요하다.

## 3. 관찰 → 가설 → 실제 추가 검증

### 사례 1. 카테고리의 금액과 상품수

- **관찰**: 금액 1위 패션은 32,940,000원, 상품수 1위는 뷰티 17개다.
- **가설**: 목록에 등록된 상품수보다 거래 수량이나 거래 단가 차이가 금액 차이에 더 직접 연결될 수 있다.
- **실제 추가 검증**: 실습 2에서 상품수·판매상품수·수량·수량가중 거래단가를 한 표로 연결했다. 패션은 수량 290개, 수량가중 단가 113,586원이다. 아래에서 `수량 × 수량가중 단가 = 금액`을 검증한다.
- **판단과 한계**: 금액의 산술 구성을 확인했다. 상품수만으로 선호도나 수요의 원인을 설명할 수 없고 노출·재고를 추가 확인해야 한다.

### 사례 2. 월별 차이

- **관찰**: 중간 월 중 2025-10의 금액이 12,547,000원으로 최대다. 2025-09은 4,519,000원이다.
- **가설**: 주문 수와 주문당 금액의 차이, 특정 카테고리의 거래 규모가 함께 영향을 주었을 수 있다.
- **실제 추가 검증**: 월별 표에서 최대 월은 17건·객단가 738,059원, 최소 월은 4건·객단가 1,129,750원이다. 월×카테고리를 다시 계산했을 때 최대 월의 최대 카테고리는 식품 3,088,000원이다.
- **판단과 한계**: 건수·객단가·구성을 실제로 나눠 확인했다. 마케팅이나 계절성의 원인은 확정할 수 없다.

### 사례 3. 고액 고객의 반복 여부

- **관찰**: 총금액 상위 고객은 주문 건수와 평균 주문금액이 같지 않다. 고객 유형별 실제 인원과 금액을 실습 5에 출력했다.
- **가설**: 고액 고객에는 반복 구매자와 고액 단건 구매자가 함께 섞일 수 있다.
- **실제 추가 검증**: 고객별 고유 주문 수를 계산해 2회 이상과 1회를 분리했다. 고액 1회 기준은 주문금액 Q75=1,098,500원이며, LLM이 제안한 고객 총금액 Q90=2,789,500원에서도 실제 인원을 비교했다.
- **판단과 한계**: 기간 내 행동은 구분했지만 미래 반복 구매나 평생 충성도는 확인하지 못했다.


In [33]:
# 사례 1: 구성 확인. 평균×수량의 부동소수점 오차는 허용한다.
assert np.allclose(category_sales.total_quantity * category_sales.quantity_weighted_unit_price,
                   category_sales.total_sales)
# 사례 2: 경계 월을 뺀 최대·최소 월과 최대 월의 카테고리 구성 확인.
full_months = monthly_sales.iloc[1:-1]
compare_months = full_months.loc[full_months.total_sales.isin([full_months.total_sales.max(),full_months.total_sales.min()])]
display(compare_months)
peak_month = full_months.loc[full_months.total_sales.idxmax(),'order_month']
display(month_category.loc[month_category.order_month.eq(peak_month)].sort_values('total_sales',ascending=False))
# 사례 3: 상위 고객도 고유 주문수와 객단가를 함께 본다.
display(customer_sales.head(10)[['customer_id','order_count','total_sales','avg_order_value','customer_group']])
print('산술 구성 검증 통과')

,order_month,total_sales,order_count,avg_order_value
2,2025-09,4519000,4,1.129750e+06
3,2025-10,12547000,17,7.380588e+05


,order_month,category,total_sales,total_quantity,order_count
25,2025-10,식품,3088000,23,5
27,2025-10,패션,2687000,20,6
26,2025-10,전자기기,1781000,22,5
21,2025-10,도서,1570000,14,5
24,2025-10,스포츠,1426000,10,3
23,2025-10,생활용품,1334000,19,6
22,2025-10,뷰티,661000,11,4


,customer_id,order_count,total_sales,avg_order_value,customer_group
1,3,3,6702000,2.234000e+06,반복 구매 (2회 이상)
7,14,4,5122000,1.280500e+06,반복 구매 (2회 이상)
56,90,3,4311000,1.437000e+06,반복 구매 (2회 이상)
46,65,6,3977000,6.628333e+05,반복 구매 (2회 이상)
65,104,2,3966000,1.983000e+06,반복 구매 (2회 이상)
58,93,4,3845000,9.612500e+05,반복 구매 (2회 이상)
11,20,4,3808000,9.520000e+05,반복 구매 (2회 이상)
27,38,4,3656000,9.140000e+05,반복 구매 (2회 이상)
19,28,4,3201000,8.002500e+05,반복 구매 (2회 이상)
54,85,4,2832000,7.080000e+05,반복 구매 (2회 이상)


산술 구성 검증 통과


## 2. 핵심 집계 결과와 검증 요약 — Evidence E04

- 동일 모집단의 집계 전 clean completed 상세 독립 계산: **134,523,000원**
- 카테고리별 합계: **134,523,000원**
- 월별 합계: **134,523,000원**
- 고객·도시·결제수단·월×카테고리 합계: 모두 **134,523,000원**
- 총합 일치 여부: **일치, 차이 0원**

**결과 관찰**: 여기서 원본 합계라는 표현은 집계 전 clean 상세를 뜻한다. dirty raw 전체의 금액을 복원한 값은 아니다.  전처리 clean을 재로딩한 뒤 날짜·금액을 다시 검증했고, 세 FK 및 many_to_one 병합과 각 금액 합계 검증이 통과했다.

**나의 해석과 판단**: 집계가 맞으려면 범주만 바꾸어도 같은 모집단이어야 한다. completed만 적용하고 날짜 제외는 월별에만 적용하는 식의 혼합을 피했다.

**업무·분석적 의미**: 같은 기준의 표를 비교하므로 카테고리·월·고객 관점의 후속 분석을 연결할 수 있다.

**한계와 추가 확인 사항**: 전처리에서 격리된 거래, 날짜 결측·미래 후보는 포함하지 않은 조건부 결과다. 부분 보존 주문 18건의 기본 금액은 남은 상세 기준이며 원래 청구총액이 아니다. 해당 주문을 전부 제외한 민감도도 위에 계산했다. 단순 제외가 매출의 진실을 복원한 것은 아니다. 제외 건수·금액은 scope 표, 전처리 사유는 Chapter05 보고서에 있다.

## 4-1. 스크립트 재실행과 저장 결과 대조 — Evidence E06

In [34]:
def report_hashes():
    return {p.name:hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(REPORT_DIR.glob('ch06_*')) if p.is_file()}
before = report_hashes()
execution = subprocess.run([sys.executable,'scripts/run_eda.py'], cwd=PROJECT_ROOT,
                           check=True, text=True, capture_output=True)
print(execution.stdout)
after = report_hashes()
assert before == after, 'Notebook와 스크립트 산출물 차이'
print('Notebook/스크립트 산출물 해시 일치:',len(after),'개')
for name in ['category_sales','monthly_sales','payment_order','city_sales','month_category']:
    saved = pd.read_csv(REPORT_DIR/f'ch06_{name}.csv', keep_default_na=False)
    pd.testing.assert_frame_equal(saved,results[name].reset_index(drop=True),check_dtype=False)
print('주요 CSV 5개 재읽기 값 일치; 종료 코드:',execution.returncode)

Chapter06 EDA 완료 — completed, 유효 주문일, 2026-10-08 기준
             항목          값
            기준일 2026-10-08
          기간 시작 2025-07-02
          기간 종료 2026-08-28
고액 1회 기준(Q75 원)  1098500.0
상세 없는 조건충족 주문 수          1
   전체 부분보존 주문 수         31
 분석 내 부분보존 주문 수         18
                 범위  상세행수  주문수      금액합계                             주석
전체 clean 상세 (매출 아님)   724  297 244374000                               
       completed 전체   430  178 137335000                               
 completed 날짜 결측 제외     6    2   1903000                    금액분석의 공통 제외
 completed 미래 후보 제외     3    1    909000                  기준일 이후를 공통 제외
          최종 분석 모집단   421  175 134523000 금액 상세가 없는 조건충족 주문 1건; 평균 분모 제외
category  total_quantity  total_sales  order_count  sold_product_count  sales_ratio_pct  quantity_weighted_unit_price
      패션             290     32940000           68                  15            24.49                 113586.206897
      식품             220     24731000           61              

## 5. 다음 분석 우선순위

1. **날짜와 거래 유효성 확인**: 격리 거래와 날짜 결측·미래 후보가 어떤 주문인지 원천 기준으로 확인한다. 실제 금액을 포함할지 결정하기 전에 품질 문제부터 해결해야 한다.
2. **월별 금액의 구성 확인**: 경계 월을 구분하고, 주문 수·객단가·월×카테고리의 변화를 시각화한다. 금액 변화가 어디에 모여 있는지 질문을 좁힐 수 있다.
3. **고객 유형과 기준의 안정성 확인**: 기간과 고액 기준을 바꿔도 반복 구매·고액 단건 구분이 얼마나 바뀌는지 확인한다. 이번 Q75/Q90 비교는 첫 점검이다.

### 가장 먼저 확인하고 싶은 이유

전처리에서 빠진 거래와 날짜가 분석 모집단을 바꾼다. 금액 순위에 대한 설명을 자세히 만들기 전에 어떤 거래를 분석했는지 확실히 하는 것이 먼저라고 판단했다.

### 현재 데이터만으로 단정할 수 없는 것

광고의 인과효과, 이익률, 재고 회전율, 고객의 평생가치, 특정 지역의 선호도, 계절성은 이 네 파일만으로 확정하지 않는다. 관측된 차이와 설명 가능한 가설까지만 정리했다.

## 최종 체크

- [x] 공식 Notebook과 가이드·답안 양식을 확인했다.
- [x] CSV 4개와 실제 컬럼·형식·날짜 범위를 확인했다.
- [x] 질문 3개 이상에 기간·상태·금액 기준·지표·검증 방법을 연결했다.
- [x] 모든 금액 분석에 같은 completed·유효 날짜 범위를 적용했다.
- [x] PK/FK·병합 행 수·총합·CSV 재읽기·스크립트 재실행을 검증했다.
- [x] 관찰·가설·실제 추가 검증·한계를 구분했다.
- [x] 추가 실습 6문항과 실제 LLM 제안 사용/수정/보류 판단을 작성했다.
- [x] 공개 수업용 합성 데이터만 사용하고 API Key·Secret을 포함하지 않았다.
- [ ] GitHub에서 최종 Notebook의 본문·출력을 확인했다.
- [ ] 최종 Notebook URL의 eTL 접수를 확인했다.

실습 실행 완료와 외부 접수 완료는 구분했다. 외부 상태는 확인 후에만 체크한다.